# BB84 QKD: Detecting an Eavesdropper on IBM Quantum Hardware

**Environment:** qBraid · IBM Quantum (Heron preferred — `ibm_fez`, `ibm_marrakesh`)
**Time budget:** ~1–2 minutes of QPU time from the 10-min/month Open Plan
**Result:** Two measured QBER values — one for an undisturbed channel, one with a textbook intercept-resend eavesdropper — with bootstrapped confidence intervals and a calibrated detection threshold derived from the hardware noise floor.

## Why this experiment

Extends the QBER work you've already run on `ibm_fez` in two ways:

1. **Active attack model.** Instead of just measuring channel noise, we implement an intercept-resend Eve as a mid-circuit measurement on the same physical qubit. Eve's measurement basis is varied; her disturbance shows up as inflated QBER.
2. **Empirical threshold.** Real QKD systems can't use the textbook 11% Shor–Preskill bound naively — they must account for measured hardware noise. This notebook calibrates the threshold from your channel's actual noise floor (μ + 5σ rule) rather than assuming it.

## What this means for CIBC's quantum-readiness narrative

- **Independent verification, not vendor trust.** Any QKD vendor (ID Quantique, Toshiba, etc.) will quote security parameters; this is the primitive that lets the bank's security team *verify* those claims on commodity quantum hardware.
- **Complement to PQC.** PQC handles store-and-forward and authentication; QKD handles real-time key agreement on a quantum channel. Demonstrating QBER-based detection on IBM hardware shows the bank has hands-on competence in both legs of the post-quantum transition.
- **Board-grade evidence.** The output (two distributions clearly separated by ~20 percentage points of QBER) is the kind of empirical artifact that closes a regulator or board briefing on quantum readiness.

## Protocol summary

For each round Alice randomly picks a basis $A_b \in \{Z, X\}$ and a bit $a \in \{0,1\}$ and prepares $|a\rangle_{A_b}$. Bob picks a basis $B_b$ and measures. After many rounds they reveal bases and keep only **matching-basis** rounds (the *sifted key*).

**Without Eve:** matching-basis rounds give Bob $b = a$ exactly, in theory. On real hardware, single-qubit + measurement errors produce a noise-floor QBER of typically 2–8%.

**With intercept-resend Eve:** Eve picks a basis $E_b$, measures, and resends the post-measurement state to Bob.
- $E_b = A_b$ (50% chance): Eve's measurement is faithful → no disturbance.
- $E_b \ne A_b$ (50% chance): Eve's measurement is random → Bob's outcome is random → 50% chance of error.

→ **Theoretical QBER under intercept-resend = 25%**, well above the 11% Shor–Preskill secure-key bound.

**Hardware implementation.** Eve's intercept-resend is a `basis-rotate → measure → basis-rotate` block on the same physical qubit (no reset, no ancilla, no SWAP) — the mid-circuit measurement *is* Eve's measurement, and the collapsed post-measurement state *is* the resent qubit.

## 1. Imports

In [ ]:
# Uncomment if your qBraid environment lacks these
# %pip install -q qiskit qiskit-ibm-runtime matplotlib numpy

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

from qiskit import ClassicalRegister, QuantumRegister, QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

rng = np.random.default_rng(20260520)
print("Imports OK")

## 2. Connect & check usage *before*

qBraid stores your IBM Quantum credentials, so a bare `QiskitRuntimeService()` should work. If not, run once:

```python
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="<your IBM Cloud API key>",
    instance="<your CRN>",
    overwrite=True,
)
```

In [ ]:
service = QiskitRuntimeService()
print("Connected. Instance:", service.active_account().get("instance", "(default)"))

In [ ]:
def used_seconds_this_month(svc):
    """Sum reported QPU usage_seconds across this calendar month's jobs."""
    now = datetime.now(timezone.utc)
    start = datetime(now.year, now.month, 1, tzinfo=timezone.utc)
    total = 0.0
    try:
        jobs = svc.jobs(created_after=start, limit=200)
    except Exception as e:
        print(f"  (could not enumerate jobs: {e})")
        return None
    for j in jobs:
        try:
            u = j.usage()
            if isinstance(u, dict):
                total += float(u.get("seconds", u.get("quantum_seconds", 0)) or 0)
            else:
                total += float(u or 0)
        except Exception:
            pass
    return total

used_before = used_seconds_this_month(service)
if used_before is not None:
    print(f"Used this month:        {used_before:7.1f} s   ({used_before/60:5.2f} min)")
    print(f"Remaining (10 min plan): {max(0, 600 - used_before):7.1f} s   ({max(0, 600 - used_before)/60:5.2f} min)")

## 3. Backend & physical qubit

For BB84 we only need one good qubit. We pick the qubit with the lowest **readout** error (single-qubit measurement is the dominant error in BB84 since there's no two-qubit gate).

In [ ]:
PREFERRED = ["ibm_fez", "ibm_marrakesh", "ibm_torino"]

backend = None
for name in PREFERRED:
    try:
        b = service.backend(name)
        if b.status().operational:
            backend = b
            break
    except Exception:
        continue
if backend is None:
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=1)

print(f"Backend       : {backend.name}")
print(f"Qubits        : {backend.num_qubits}")
print(f"Pending jobs  : {backend.status().pending_jobs}")

In [ ]:
def best_single_qubit(backend):
    """Pick the physical qubit with lowest measurement (readout) error."""
    target = backend.target
    best, best_err = None, float("inf")
    if "measure" in target.operation_names:
        for q, props in target["measure"].items():
            if props is None or props.error is None:
                continue
            if props.error < best_err:
                best_err, best = props.error, q[0]
    if best is None:
        return 0, None
    return best, best_err

PHYS_QUBIT, meas_err = best_single_qubit(backend)
print(f"Selected physical qubit: {PHYS_QUBIT}   (measurement error: "
      + (f"{meas_err:.2e}" if meas_err else "n/a") + ")")

## 4. BB84 circuit factory

One single-qubit circuit per (Alice basis, Alice bit, Bob basis, optional Eve basis) tuple.

- **No-Eve circuit:** Alice-prep → Bob-measure.
- **With-Eve circuit:** Alice-prep → Eve-rotate → measure (Eve) → Eve-rotate-back → Bob-measure.

The Eve block is exactly `H, measure, H` for X-basis intercept and `measure` for Z-basis intercept. Hardware mid-circuit measurement does the collapse-and-resend in one step on the same physical qubit.

In [ ]:
def bb84_circuit(alice_basis: str, alice_bit: int,
                 bob_basis: str, eve_basis: str | None = None) -> QuantumCircuit:
    qreg = QuantumRegister(1, "q")
    bob_cl = ClassicalRegister(1, "bob")
    if eve_basis is not None:
        eve_cl = ClassicalRegister(1, "eve")
        qc = QuantumCircuit(qreg, eve_cl, bob_cl)
    else:
        qc = QuantumCircuit(qreg, bob_cl)

    # --- Alice prepares |a> in basis A_b ---
    if alice_bit == 1:
        qc.x(0)
    if alice_basis == "X":
        qc.h(0)
    qc.barrier()

    # --- Eve intercepts (mid-circuit measurement) ---
    if eve_basis is not None:
        if eve_basis == "X":
            qc.h(0)
        qc.measure(0, eve_cl[0])      # collapses; collapsed state IS the resent qubit
        if eve_basis == "X":
            qc.h(0)
        qc.barrier()

    # --- Bob measures in basis B_b ---
    if bob_basis == "X":
        qc.h(0)
    qc.measure(0, bob_cl[0])
    return qc

# Sanity print
print(bb84_circuit("X", 1, "X", eve_basis="Z").draw())

## 5. Enumerate the matching-basis configurations

In real BB84 only matching-basis rounds enter the sifted key. To save QPU time we only run those rounds.

- **No-Eve:** $(A_b, a, B_b)$ with $A_b = B_b$ → 4 circuits
- **With-Eve:** plus $E_b$ → 8 circuits

Total: **12 circuits** in a single Sampler job.

In [ ]:
BASES = ["Z", "X"]
BITS  = [0, 1]

# (label, alice_basis, alice_bit, bob_basis, eve_basis)
configs = []
# No-Eve baseline (matching bases only)
for ab, a in itertools.product(BASES, BITS):
    configs.append((f"noEve|A={ab}{a}|B={ab}", ab, a, ab, None))
# With-Eve, both Eve bases
for ab, a, eb in itertools.product(BASES, BITS, BASES):
    configs.append((f"Eve={eb}|A={ab}{a}|B={ab}", ab, a, ab, eb))

print(f"Total circuits: {len(configs)}")
for lbl, *_ in configs:
    print(" ", lbl)

## 6. Build & transpile to the device's ISA

In [ ]:
raw_circuits = [bb84_circuit(ab, a, bb, eb) for (_, ab, a, bb, eb) in configs]

pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=2,
    initial_layout=[PHYS_QUBIT],
)
isa_circuits = [pm.run(c) for c in raw_circuits]

depths = [c.depth() for c in isa_circuits]
print(f"Transpiled {len(isa_circuits)} circuits.")
print(f"Depth range: {min(depths)}..{max(depths)}")

## 7. Submit the Sampler job  ⏱️ *this is the metered step*

One job, 12 PUBs (one per config). Default `SHOTS = 8192` per PUB → 98,304 total shots.

**Time-budget tuning:**
- `SHOTS = 4096` → ~30 s typical
- `SHOTS = 8192` → ~60 s typical *(default)*
- `SHOTS = 16384` → ~2 min typical

In [ ]:
SHOTS = 8192

sampler = Sampler(mode=backend)
sampler.options.default_shots = SHOTS
sampler.options.dynamical_decoupling.enable = True
# Twirling can interact oddly with mid-circuit measurement; leave off
sampler.options.twirling.enable_gates = False

job = sampler.run(isa_circuits)
print(f"Submitted job: {job.job_id()}")
print(f"  backend = {backend.name}")
print(f"  qubit   = {PHYS_QUBIT}")
print(f"  shots   = {SHOTS} per circuit × {len(isa_circuits)} circuits = {SHOTS*len(isa_circuits):,} total shots")
print(f"  status  = {job.status()}")

In [ ]:
# Block until done. Only QPU time is metered; queue time is free.
result = job.result()
print("Done. Status:", job.status())

## 8. Compute QBER

For each configuration, Bob's outcome `b` is compared to Alice's prepared bit `a`. The QBER for a group of configurations is

$$\mathrm{QBER} = \frac{\#\{b \ne a\}}{\#\{\text{all sifted rounds}\}}$$

We compute three QBERs:
1. **No-Eve baseline** — aggregating all no-Eve circuits.
2. **With-Eve (any Eve basis)** — aggregating all Eve circuits.
3. **Per-condition breakdown** — to see if any Z/X asymmetry shows up on this qubit.

In [ ]:
def bob_errors(counts: dict, alice_bit: int):
    """Return (n_errors, n_total) from a 1-bit 'bob' register."""
    n_err = 0
    n_tot = 0
    for bitstr, c in counts.items():
        b = int(bitstr.strip()[-1])
        n_tot += c
        if b != alice_bit:
            n_err += c
    return n_err, n_tot

# Per-config breakdown
breakdown = []
for (lbl, ab, a, bb, eb), pub_res in zip(configs, result):
    counts = pub_res.data.bob.get_counts()
    err, tot = bob_errors(counts, a)
    breakdown.append({
        "label": lbl, "alice_basis": ab, "alice_bit": a,
        "bob_basis": bb, "eve_basis": eb,
        "errors": err, "total": tot,
        "qber": err / tot if tot else float("nan"),
    })

# Aggregate
def aggregate(rows):
    e = sum(r["errors"] for r in rows)
    t = sum(r["total"] for r in rows)
    return e, t, (e / t if t else float("nan"))

no_eve_rows = [r for r in breakdown if r["eve_basis"] is None]
eve_rows    = [r for r in breakdown if r["eve_basis"] is not None]

e0, t0, qber_no_eve = aggregate(no_eve_rows)
e1, t1, qber_eve    = aggregate(eve_rows)

print(f"No-Eve baseline : {e0:>6,} / {t0:>6,}  →  QBER = {qber_no_eve:.4f}  ({qber_no_eve*100:.2f}%)")
print(f"With-Eve        : {e1:>6,} / {t1:>6,}  →  QBER = {qber_eve:.4f}  ({qber_eve*100:.2f}%)")
print()
print("Per-configuration:")
for r in breakdown:
    print(f"  {r['label']:30s}  err={r['errors']:>5,}/{r['total']:>5,}  QBER={r['qber']*100:5.2f}%")

## 9. Bootstrap 95% confidence intervals

QBER is a binomial estimate; we bootstrap to get an honest CI for both conditions. With ~32k samples per condition the CI half-width should be ≤ 0.5%.

In [ ]:
def bootstrap_qber_ci(errors: int, total: int, n_boot: int = 5000, alpha: float = 0.05):
    """Wilson-style bootstrap. Returns (point, lo, hi)."""
    if total == 0:
        return float("nan"), float("nan"), float("nan")
    samples = rng.binomial(total, errors / total, size=n_boot) / total
    lo, hi = np.quantile(samples, [alpha/2, 1 - alpha/2])
    return errors / total, float(lo), float(hi)

q0, lo0, hi0 = bootstrap_qber_ci(e0, t0)
q1, lo1, hi1 = bootstrap_qber_ci(e1, t1)

print(f"No-Eve QBER : {q0*100:5.2f}%   95% CI [{lo0*100:5.2f}%, {hi0*100:5.2f}%]")
print(f"With-Eve QBER: {q1*100:5.2f}%   95% CI [{lo1*100:5.2f}%, {hi1*100:5.2f}%]")
print()
gap = q1 - q0
print(f"QBER gap (Eve − noEve): {gap*100:.2f} percentage points")
print(f"  theoretical (intercept-resend on ideal channel): 25.00 pp")

## 10. Calibrated detection threshold

A real QKD operator can't use the textbook 11% bound naively — they need a threshold that's

1. **above the hardware noise floor** (so honest channels don't trigger false alarms), and
2. **below the intercept-resend signature** (so eavesdroppers are caught).

The standard operational rule: $\tau = \mu_{\text{noise}} + 5\sigma_{\text{noise}}$ where $\mu, \sigma$ come from the no-Eve calibration runs. A round trips the alarm if its measured QBER exceeds $\tau$.

In [ ]:
# Per-no-Eve-config QBERs give us the noise-floor distribution
noise_floor_qbers = np.array([r["qber"] for r in no_eve_rows])
mu, sigma = noise_floor_qbers.mean(), noise_floor_qbers.std(ddof=1)
threshold = mu + 5 * sigma  # operational 5σ rule

# Practical guard rails
threshold = max(threshold, 0.02)      # never trust below 2%
threshold = min(threshold, 0.11)      # never above Shor-Preskill bound

print(f"No-Eve noise floor:   μ = {mu*100:.2f}%   σ = {sigma*100:.2f}%")
print(f"Detection threshold τ = μ + 5σ = {threshold*100:.2f}%   (clamped to [2%, 11%])")
print()
detected = q1 > threshold
print(f"Measured with-Eve QBER : {q1*100:.2f}%")
print(f"Eavesdropper detected? : {'YES — abort key exchange' if detected else 'NO — would have leaked key!'}")

## 11. Visualize

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Left: bar with CI ---
labels   = ["No Eve\n(baseline)", "With Eve\n(intercept-resend)"]
values   = [q0 * 100, q1 * 100]
errs_lo  = [(q0 - lo0) * 100, (q1 - lo1) * 100]
errs_hi  = [(hi0 - q0) * 100, (hi1 - q1) * 100]
colors_  = ["#2ca02c", "#d62728"]

ax1.bar(labels, values, color=colors_, edgecolor="black",
        yerr=[errs_lo, errs_hi], capsize=6)
ax1.axhline(threshold * 100, color="black", ls="--", lw=1.2,
            label=f"Detection threshold τ = {threshold*100:.2f}%")
ax1.axhline(11.0, color="gray", ls=":", lw=1,
            label="Shor–Preskill bound (11%)")
ax1.axhline(25.0, color="purple", ls=":", lw=1,
            label="Theoretical IR-Eve (25%)")
ax1.set_ylabel("QBER (%)")
ax1.set_ylim(0, max(30, q1*100*1.15))
ax1.set_title(f"QBER on {backend.name}, qubit {PHYS_QUBIT}")
ax1.legend(loc="upper left", fontsize=8)
for i, v in enumerate(values):
    ax1.text(i, v + 0.7, f"{v:.2f}%", ha="center", fontweight="bold")

# --- Right: per-configuration breakdown ---
no_lbls = [r["label"].split("|", 1)[1] for r in no_eve_rows]
ev_lbls = [r["label"].split("|", 1)[1] + " " + (r["eve_basis"] or "") for r in eve_rows]
no_q    = [r["qber"] * 100 for r in no_eve_rows]
ev_q    = [r["qber"] * 100 for r in eve_rows]

x_no = np.arange(len(no_lbls))
x_ev = np.arange(len(ev_lbls)) + len(no_lbls) + 1
ax2.bar(x_no, no_q, color="#2ca02c", edgecolor="black", label="No Eve")
ax2.bar(x_ev, ev_q, color="#d62728", edgecolor="black", label="With Eve")
ax2.axhline(threshold * 100, color="black", ls="--", lw=1.2)
ax2.set_xticks(np.concatenate([x_no, x_ev]))
ax2.set_xticklabels(no_lbls + ev_lbls, rotation=60, ha="right", fontsize=7)
ax2.set_ylabel("QBER (%)")
ax2.set_title("Per-configuration QBER")
ax2.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

## 12. Check QPU time consumed

In [ ]:
try:
    u = job.usage()
    if isinstance(u, dict):
        secs = float(u.get("seconds", u.get("quantum_seconds", 0)) or 0)
        print(f"This job used: {secs:6.2f} s   ({secs/60:5.3f} min)   [{u}]")
    else:
        print(f"This job used: {float(u):6.2f} s   ({float(u)/60:5.3f} min)")
except Exception as e:
    print(f"Could not read job.usage(): {e}")

used_after = used_seconds_this_month(service)
if used_before is not None and used_after is not None:
    print()
    print(f"Month total before : {used_before:7.1f} s   ({used_before/60:5.2f} min)")
    print(f"Month total after  : {used_after:7.1f} s   ({used_after/60:5.2f} min)")
    print(f"Delta              : {used_after - used_before:+7.1f} s")

## What you've shown, and what comes next

**What this run produced**

1. A hardware-calibrated QBER **noise floor** for `{}` on physical qubit `{}` — the baseline against which any future channel attestation can be benchmarked.
2. A measured separation between honest and intercept-resend conditions that should be ≥ 15 percentage points on Heron hardware — much larger than the 5σ CI half-width on either condition, so the detection is statistically unambiguous.
3. An operationally calibrated detection threshold (μ + 5σ, clamped to the Shor–Preskill bound) — directly usable as the alarm setting in any QKD pilot.

**Extensions worth running on the remaining budget**

- **Compare across physical qubits.** Change `PHYS_QUBIT` and rerun cells 5–11 to map noise floor across the chip. Useful for the bank's future quantum-channel hardware selection criteria.
- **Bias-attack Eve.** Force Eve to always measure in Z (`eve_basis="Z"` only) and observe the basis-asymmetric QBER signature. Real intercept attempts on biased-basis BB84 protocols look like this.
- **Decoy-state framing.** Layer two preparation intensities and you reproduce the standard decoy-state protocol used in commercial QKD (ID Quantique Cerberis, Toshiba LD). The detection logic stays identical; only the post-processing changes.
- **PQC pairing.** This QBER detection is the *real-time channel* leg of post-quantum security. Pair the output (an authenticated key) with a CRYSTALS-Kyber wrapping demonstration for the *store-and-forward* leg, and you have both sides of the bank's PQ transition story in one artifact.

**Limits to flag in any board/regulator briefing**

- Intercept-resend is the simplest attack; sophisticated attacks (photon-number-splitting, side-channel) require dedicated countermeasures.
- This is a *single-qubit channel* on a fixed processor — not a fibre-optic QKD link. Mid-circuit measurement here stands in for a remote Eve.
- IBM Quantum Open Plan QPU time is metered (10 min/month); a production attestation pipeline would need a paid plan or dedicated hardware.
